# SpottingSalmon
YOLO model logging script
- To register the final model as an MLFlow model


In [0]:
%pip install --quiet numpy==1.26.4  mlflow ultralytics

In [0]:
import mlflow
import mlflow.pyfunc
import ultralytics
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, TensorSpec, DataType, ColSpec
import numpy as np
from ultralytics import YOLO
from pyspark.sql import functions as F

In [0]:
from ultralytics import settings

# Update a setting
settings.update({"mlflow": True})

# Reset settings to default values
settings.reset()

### 1. Model logging

In [0]:
# Run ID and artifact relative path
run_id = "95806984982941ebbedc746d54719cb7"
artifact_path = "weights/best.pt"

#model_path = f"runs:/{run_id}/{artifact_path}"
# Download to local temp dir
model_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path=artifact_path
)

print(f"Model downloaded to: {model_path}")

# Load the model
model = YOLO(model_path)

In [0]:
class YOLOPyfuncModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        from ultralytics import YOLO
        self.model = YOLO(context.artifacts["checkpoint"])

    def predict(self, context, input_data):
        """
        input_data is a pandas.DataFrame containing:
        - image path strings, or
        - base64 image blobs, or
        - bytes, depending on your use case
        """
        results = []
        for img in input_data:
            r = self.model(img)
            results.append(r[0].tojson())  # simple output format
        return results


In [0]:
input_schema = Schema([
    ColSpec("string", name="fish")
])

output_schema = Schema([
    ColSpec("string")   
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)


In [0]:
with mlflow.start_run(run_id="95806984982941ebbedc746d54719cb7"):

    mlflow.pyfunc.log_model(
        artifact_path="fish_yolo_m4",
        python_model=YOLOPyfuncModel(),
        artifacts={"checkpoint": model_path},
        signature=signature
    )

### Register model

In [0]:
mlflow.set_registry_uri("databricks-uc")

In [0]:
mlflow.register_model(
    model_uri="runs:/95806984982941ebbedc746d54719cb7/fish_yolo_m4",
    name="prd_dash_lab.dash_data_science_unrestricted.salmon_model_test"
)

### Test registered model 

In [0]:
model = mlflow.pyfunc.load_model("models:/prd_dash_lab.dash_data_science_unrestricted.salmon_model_test/2")

In [0]:
import pandas as pd

df = pd.DataFrame({"image": ["/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/Fish (1).jpg"]})


In [0]:
import pyspark.sql.functions as F

img_df = spark.read.format("binaryFile").load(
    "dbfs:/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/Fish (1).jpg").withColumn("image", F.col("content").cast("binary"))

sample = img_df.limit(1).collect()[0]
fish = sample.image


In [0]:
display(fish)

In [0]:
results = model.predict(fish)
